#### Read all per-config agg500 metadata CSVs

#### Concatenate into a unified DataFrame

#### Stratify 5 folds by (alpha, q0) using sklearn

#### Save unified CSV + per-fold train/test CSVs

Cell 1 – Imports and paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# ------------------------------------------------------------------
# Adjust these to your setup
# ------------------------------------------------------------------
# LOCAL_ROOT = "/mnt/Research/arsalan/wsu_grid_rs_grp_mljet/ml-jet-ext/"
LOCAL_ROOT="/home/arsalan/wsu_grid_rs_grp_mljet/"
# Root where the aggregated 500-jet dataset is stored
AGG_OUTPUT_ROOT = Path(
    f"{LOCAL_ROOT}pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500"
).expanduser()

AGG_METADATA_DIR = AGG_OUTPUT_ROOT / "metadata"

print("AGG_OUTPUT_ROOT:", AGG_OUTPUT_ROOT)
print("AGG_METADATA_DIR:", AGG_METADATA_DIR)
assert AGG_METADATA_DIR.exists(), "AGG_METADATA_DIR does not exist"


AGG_OUTPUT_ROOT: /home/arsalan/wsu_grid_rs_grp_mljet/pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500
AGG_METADATA_DIR: /home/arsalan/wsu_grid_rs_grp_mljet/pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500/metadata


Cell 2 – Read all per-config agg500 metadata CSVs and concatenate

In [2]:
# Find all per-config agg500 metadata CSVs
agg_meta_files = sorted(AGG_METADATA_DIR.glob("*_agg500_metadata.csv"))
print("Found metadata files:", len(agg_meta_files))
for p in agg_meta_files:
    print("  ", p.name)

if not agg_meta_files:
    raise FileNotFoundError(f"No *_agg500_metadata.csv files found in {AGG_METADATA_DIR}")


Found metadata files: 35
   c0005_alpha0p20_q_1p0_agg500_metadata.csv
   c0005_alpha0p20_q_1p5_agg500_metadata.csv
   c0005_alpha0p20_q_2p0_agg500_metadata.csv
   c0005_alpha0p20_q_2p5_agg500_metadata.csv
   c0005_alpha0p20_q_3p0_agg500_metadata.csv
   c0005_alpha0p25_q_1p0_agg500_metadata.csv
   c0005_alpha0p25_q_1p5_agg500_metadata.csv
   c0005_alpha0p25_q_2p0_agg500_metadata.csv
   c0005_alpha0p25_q_2p5_agg500_metadata.csv
   c0005_alpha0p25_q_3p0_agg500_metadata.csv
   c0005_alpha0p30_q_1p0_agg500_metadata.csv
   c0005_alpha0p30_q_1p5_agg500_metadata.csv
   c0005_alpha0p30_q_2p0_agg500_metadata.csv
   c0005_alpha0p30_q_2p5_agg500_metadata.csv
   c0005_alpha0p30_q_3p0_agg500_metadata.csv
   c0005_alpha0p35_q_1p0_agg500_metadata.csv
   c0005_alpha0p35_q_1p5_agg500_metadata.csv
   c0005_alpha0p35_q_2p0_agg500_metadata.csv
   c0005_alpha0p35_q_2p5_agg500_metadata.csv
   c0005_alpha0p35_q_3p0_agg500_metadata.csv
   c0005_alpha0p40_q_1p0_agg500_metadata.csv
   c0005_alpha0p40_q_1p5_agg50

In [3]:

for p in agg_meta_files:
    df_cfg = pd.read_csv(p)
    print("Total aggregated samples:", len(df_cfg))


Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggregated samples: 400
Total aggr

In [14]:
# Load and concatenate all metadata
dfs = []
for p in agg_meta_files:
    df_cfg = pd.read_csv(p)
    dfs.append(df_cfg)

In [16]:
dfs[0].head()

,agg_file_path,alpha,config,q0,source_file_paths
0,c0005_alpha0p20_q_1p0/agg500_event_0000000.npy,0.2,c0005_alpha0p20_q_1p0,1.0,c0005_alpha0p20_q_1p0_unfolded/seed_142/event_...
1,c0005_alpha0p20_q_1p0/agg500_event_0000001.npy,0.2,c0005_alpha0p20_q_1p0,1.0,c0005_alpha0p20_q_1p0_unfolded/seed_129/event_...
2,c0005_alpha0p20_q_1p0/agg500_event_0000002.npy,0.2,c0005_alpha0p20_q_1p0,1.0,c0005_alpha0p20_q_1p0_unfolded/seed_136/event_...
3,c0005_alpha0p20_q_1p0/agg500_event_0000003.npy,0.2,c0005_alpha0p20_q_1p0,1.0,c0005_alpha0p20_q_1p0_unfolded/seed_059/event_...
4,c0005_alpha0p20_q_1p0/agg500_event_0000004.npy,0.2,c0005_alpha0p20_q_1p0,1.0,c0005_alpha0p20_q_1p0_unfolded/seed_156/event_...


In [17]:
# Maps (float version — use this if alpha/q0 columns are numeric)
alpha_map = {0.2: 0, 0.25: 1, 0.3: 2, 0.35: 3, 0.4: 4, 0.45: 5, 0.5: 6}
q0_map    = {1.0: 0, 1.5: 1, 2.0: 2, 2.5: 3, 3.0: 4}

for df in dfs:
    # keep literal values
    df["alpha_val"] = df["alpha"]
    df["q0_val"]    = df["q0"]

    # mapped integer indices
    df["alpha"] = df["alpha_val"].map(alpha_map)
    df["q0"]    = df["q0_val"].map(q0_map)



In [22]:
dfs[34].head()

,agg_file_path,alpha,config,q0,source_file_paths,alpha_val,q0_val
0,c0005_alpha0p50_q_3p0/agg500_event_0000000.npy,6,c0005_alpha0p50_q_3p0,4,c0005_alpha0p50_q_3p0_unfolded/seed_031/event_...,0.5,3.0
1,c0005_alpha0p50_q_3p0/agg500_event_0000001.npy,6,c0005_alpha0p50_q_3p0,4,c0005_alpha0p50_q_3p0_unfolded/seed_022/event_...,0.5,3.0
2,c0005_alpha0p50_q_3p0/agg500_event_0000002.npy,6,c0005_alpha0p50_q_3p0,4,c0005_alpha0p50_q_3p0_unfolded/seed_199/event_...,0.5,3.0
3,c0005_alpha0p50_q_3p0/agg500_event_0000003.npy,6,c0005_alpha0p50_q_3p0,4,c0005_alpha0p50_q_3p0_unfolded/seed_176/event_...,0.5,3.0
4,c0005_alpha0p50_q_3p0/agg500_event_0000004.npy,6,c0005_alpha0p50_q_3p0,4,c0005_alpha0p50_q_3p0_unfolded/seed_150/event_...,0.5,3.0


In [42]:
# add another column to each dfs call it eloss, if q0=0 (because this is the maped index now) then eloss=0 otherwise eloss=1
for df in dfs:
    df['eloss'] = np.where(df['q0'] == 0, 0, 1)

In [49]:
dfs[10].head()

,agg_file_path,alpha,config,q0,source_file_paths,alpha_val,q0_val,eloss
0,c0005_alpha0p30_q_1p0/agg500_event_0000000.npy,2,c0005_alpha0p30_q_1p0,0,c0005_alpha0p30_q_1p0_unfolded/seed_071/event_...,0.3,1.0,0
1,c0005_alpha0p30_q_1p0/agg500_event_0000001.npy,2,c0005_alpha0p30_q_1p0,0,c0005_alpha0p30_q_1p0_unfolded/seed_114/event_...,0.3,1.0,0
2,c0005_alpha0p30_q_1p0/agg500_event_0000002.npy,2,c0005_alpha0p30_q_1p0,0,c0005_alpha0p30_q_1p0_unfolded/seed_148/event_...,0.3,1.0,0
3,c0005_alpha0p30_q_1p0/agg500_event_0000003.npy,2,c0005_alpha0p30_q_1p0,0,c0005_alpha0p30_q_1p0_unfolded/seed_008/event_...,0.3,1.0,0
4,c0005_alpha0p30_q_1p0/agg500_event_0000004.npy,2,c0005_alpha0p30_q_1p0,0,c0005_alpha0p30_q_1p0_unfolded/seed_136/event_...,0.3,1.0,0


In [50]:
all_df = pd.concat(dfs, ignore_index=True)
print("Total aggregated samples:", len(all_df))
# Add agg_id column: agg_00000 ... agg_XXXXX
all_df["agg_id"] = [
    f"agg_{i:05d}" for i in range(len(all_df))
]

all_df.head()

Total aggregated samples: 14000


,agg_file_path,alpha,config,q0,source_file_paths,alpha_val,q0_val,eloss,agg_id
0,c0005_alpha0p20_q_1p0/agg500_event_0000000.npy,0,c0005_alpha0p20_q_1p0,0,c0005_alpha0p20_q_1p0_unfolded/seed_142/event_...,0.2,1.0,0,agg_00000
1,c0005_alpha0p20_q_1p0/agg500_event_0000001.npy,0,c0005_alpha0p20_q_1p0,0,c0005_alpha0p20_q_1p0_unfolded/seed_129/event_...,0.2,1.0,0,agg_00001
2,c0005_alpha0p20_q_1p0/agg500_event_0000002.npy,0,c0005_alpha0p20_q_1p0,0,c0005_alpha0p20_q_1p0_unfolded/seed_136/event_...,0.2,1.0,0,agg_00002
3,c0005_alpha0p20_q_1p0/agg500_event_0000003.npy,0,c0005_alpha0p20_q_1p0,0,c0005_alpha0p20_q_1p0_unfolded/seed_059/event_...,0.2,1.0,0,agg_00003
4,c0005_alpha0p20_q_1p0/agg500_event_0000004.npy,0,c0005_alpha0p20_q_1p0,0,c0005_alpha0p20_q_1p0_unfolded/seed_156/event_...,0.2,1.0,0,agg_00004


In [52]:
# Current columns
cols = list(all_df.columns)

# Desired front columns
front = ["agg_id", "alpha", "q0","eloss","agg_file_path","alpha_val","q0_val"]

# Everything else, in original order but excluding the ones we moved to the front
rest = [c for c in cols if c not in front]

# Reorder
all_df = all_df[front + rest]

all_df.head()


,agg_id,alpha,q0,eloss,agg_file_path,alpha_val,q0_val,config,source_file_paths
0,agg_00000,0,0,0,c0005_alpha0p20_q_1p0/agg500_event_0000000.npy,0.2,1.0,c0005_alpha0p20_q_1p0,c0005_alpha0p20_q_1p0_unfolded/seed_142/event_...
1,agg_00001,0,0,0,c0005_alpha0p20_q_1p0/agg500_event_0000001.npy,0.2,1.0,c0005_alpha0p20_q_1p0,c0005_alpha0p20_q_1p0_unfolded/seed_129/event_...
2,agg_00002,0,0,0,c0005_alpha0p20_q_1p0/agg500_event_0000002.npy,0.2,1.0,c0005_alpha0p20_q_1p0,c0005_alpha0p20_q_1p0_unfolded/seed_136/event_...
3,agg_00003,0,0,0,c0005_alpha0p20_q_1p0/agg500_event_0000003.npy,0.2,1.0,c0005_alpha0p20_q_1p0,c0005_alpha0p20_q_1p0_unfolded/seed_059/event_...
4,agg_00004,0,0,0,c0005_alpha0p20_q_1p0/agg500_event_0000004.npy,0.2,1.0,c0005_alpha0p20_q_1p0,c0005_alpha0p20_q_1p0_unfolded/seed_156/event_...


In [53]:
all_df.tail()

,agg_id,alpha,q0,eloss,agg_file_path,alpha_val,q0_val,config,source_file_paths
13995,agg_13995,6,4,1,c0005_alpha0p50_q_3p0/agg500_event_0000395.npy,0.5,3.0,c0005_alpha0p50_q_3p0,c0005_alpha0p50_q_3p0_unfolded/seed_041/event_...
13996,agg_13996,6,4,1,c0005_alpha0p50_q_3p0/agg500_event_0000396.npy,0.5,3.0,c0005_alpha0p50_q_3p0,c0005_alpha0p50_q_3p0_unfolded/seed_112/event_...
13997,agg_13997,6,4,1,c0005_alpha0p50_q_3p0/agg500_event_0000397.npy,0.5,3.0,c0005_alpha0p50_q_3p0,c0005_alpha0p50_q_3p0_unfolded/seed_066/event_...
13998,agg_13998,6,4,1,c0005_alpha0p50_q_3p0/agg500_event_0000398.npy,0.5,3.0,c0005_alpha0p50_q_3p0,c0005_alpha0p50_q_3p0_unfolded/seed_097/event_...
13999,agg_13999,6,4,1,c0005_alpha0p50_q_3p0/agg500_event_0000399.npy,0.5,3.0,c0005_alpha0p50_q_3p0,c0005_alpha0p50_q_3p0_unfolded/seed_068/event_...


Cell 3 – Sanity check that alpha and q0 columns exist

In [54]:
assert "eloss" in all_df.columns, "'eloss' column missing in aggregated metadata"
assert "alpha" in all_df.columns, "'alpha' column missing in aggregated metadata"
assert "q0"   in all_df.columns, "'q0' column missing in aggregated metadata"

print("Unique eloss values:", sorted(all_df["eloss"].unique()))
print("Unique alpha values:", sorted(all_df["alpha"].unique()))
print("Unique q0 values:", sorted(all_df["q0"].unique()))


Unique eloss values: [np.int64(0), np.int64(1)]
Unique alpha values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Unique q0 values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


Cell 4 – Build unified CSV (optional shuffle) before folds

In [55]:
# Optional: shuffle once for good measure (StratifiedKFold will also shuffle)
all_df = all_df.sample(frac=1.0, random_state=0).reset_index(drop=True)

unified_csv = AGG_METADATA_DIR / "c0005_agg500_14000jets_metadata.csv"
all_df.to_csv(unified_csv, index=False)
print(f"Saved unified (pre-fold) CSV to: {unified_csv}")


Saved unified (pre-fold) CSV to: /home/arsalan/wsu_grid_rs_grp_mljet/pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500/metadata/c0005_agg500_14000jets_metadata.csv


In [56]:
all_df.head()

,agg_id,alpha,q0,eloss,agg_file_path,alpha_val,q0_val,config,source_file_paths
0,agg_06038,3,0,0,c0005_alpha0p35_q_1p0/agg500_event_0000038.npy,0.35,1.0,c0005_alpha0p35_q_1p0,c0005_alpha0p35_q_1p0_unfolded/seed_193/event_...
1,agg_10758,5,1,1,c0005_alpha0p45_q_1p5/agg500_event_0000358.npy,0.45,1.5,c0005_alpha0p45_q_1p5,c0005_alpha0p45_q_1p5_unfolded/seed_091/event_...
2,agg_13882,6,4,1,c0005_alpha0p50_q_3p0/agg500_event_0000282.npy,0.50,3.0,c0005_alpha0p50_q_3p0,c0005_alpha0p50_q_3p0_unfolded/seed_027/event_...
3,agg_02760,1,1,1,c0005_alpha0p25_q_1p5/agg500_event_0000360.npy,0.25,1.5,c0005_alpha0p25_q_1p5,c0005_alpha0p25_q_1p5_unfolded/seed_169/event_...
4,agg_10778,5,1,1,c0005_alpha0p45_q_1p5/agg500_event_0000378.npy,0.45,1.5,c0005_alpha0p45_q_1p5,c0005_alpha0p45_q_1p5_unfolded/seed_123/event_...


Cell 5 – Stratified 5-fold split by (alpha, q0) using sklearn

In [57]:
N_FOLDS = 5
RANDOM_STATE = 0

# Build a combined label "alpha_q0" for stratification
strata = all_df[["alpha", "q0"]].astype(str).agg("_".join, axis=1)

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Initialize fold column
all_df["fold"] = -1

for fold_id, (_, test_idx) in enumerate(skf.split(X=all_df, y=strata)):
    all_df.loc[test_idx, "fold"] = fold_id

# Sanity check
assert (all_df["fold"] >= 0).all(), "Some rows did not get a fold assignment"

print("Fold counts:")
print(all_df["fold"].value_counts().sort_index())


Fold counts:
fold
0    2800
1    2800
2    2800
3    2800
4    2800
Name: count, dtype: int64


Optional: inspect how (alpha, q0) is distributed across folds.

In [58]:
alpha_q0_fold_counts = all_df.groupby(["fold", "alpha", "q0", "eloss"]).size()
alpha_q0_fold_counts.unstack(level=0, fill_value=0)


fold             0   1   2   3   4
alpha q0 eloss                    
0     0  0      80  80  80  80  80
      1  1      80  80  80  80  80
      2  1      80  80  80  80  80
      3  1      80  80  80  80  80
      4  1      80  80  80  80  80
1     0  0      80  80  80  80  80
      1  1      80  80  80  80  80
      2  1      80  80  80  80  80
      3  1      80  80  80  80  80
      4  1      80  80  80  80  80
2     0  0      80  80  80  80  80
      1  1      80  80  80  80  80
      2  1      80  80  80  80  80
      3  1      80  80  80  80  80
      4  1      80  80  80  80  80
3     0  0      80  80  80  80  80
      1  1      80  80  80  80  80
      2  1      80  80  80  80  80
      3  1      80  80  80  80  80
      4  1      80  80  80  80  80
4     0  0      80  80  80  80  80
      1  1      80  80  80  80  80
      2  1      80  80  80  80  80
      3  1      80  80  80  80  80
      4  1      80  80  80  80  80
5     0  0      80  80  80  80  80
      1  1      80  80  80  80  80
      2  1      80  80  80  80  80
      3  1      80  80  80  80  80
      4  1      80  80  80  80  80
6     0  0      80  80  80  80  80
      1  1      80  80  80  80  80
      2  1      80  80  80  80  80
      3  1      80  80  80  80  80
      4  1      80  80  80  80  80

In [59]:
all_df.head()

,agg_id,alpha,q0,eloss,agg_file_path,alpha_val,q0_val,config,source_file_paths,fold
0,agg_06038,3,0,0,c0005_alpha0p35_q_1p0/agg500_event_0000038.npy,0.35,1.0,c0005_alpha0p35_q_1p0,c0005_alpha0p35_q_1p0_unfolded/seed_193/event_...,1
1,agg_10758,5,1,1,c0005_alpha0p45_q_1p5/agg500_event_0000358.npy,0.45,1.5,c0005_alpha0p45_q_1p5,c0005_alpha0p45_q_1p5_unfolded/seed_091/event_...,4
2,agg_13882,6,4,1,c0005_alpha0p50_q_3p0/agg500_event_0000282.npy,0.50,3.0,c0005_alpha0p50_q_3p0,c0005_alpha0p50_q_3p0_unfolded/seed_027/event_...,2
3,agg_02760,1,1,1,c0005_alpha0p25_q_1p5/agg500_event_0000360.npy,0.25,1.5,c0005_alpha0p25_q_1p5,c0005_alpha0p25_q_1p5_unfolded/seed_169/event_...,3
4,agg_10778,5,1,1,c0005_alpha0p45_q_1p5/agg500_event_0000378.npy,0.45,1.5,c0005_alpha0p45_q_1p5,c0005_alpha0p45_q_1p5_unfolded/seed_123/event_...,2


Cell 6 – Save unified stratified CSV with fold assignments

In [60]:
unified_strat_csv = AGG_METADATA_DIR / "c0005_alpha_q0_agg500_14000jets_5fold_stratified_metadata.csv"
all_df.to_csv(unified_strat_csv, index=False)
print(f"Saved unified stratified CSV to: {unified_strat_csv}")


Saved unified stratified CSV to: /home/arsalan/wsu_grid_rs_grp_mljet/pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500/metadata/c0005_alpha_q0_agg500_14000jets_5fold_stratified_metadata.csv


Cell 7 – Create per-fold train/test CSVs

In [61]:
for f in range(N_FOLDS):
    fold_test = all_df[all_df["fold"] == f].reset_index(drop=True)
    fold_train = all_df[all_df["fold"] != f].reset_index(drop=True)

    train_csv = AGG_METADATA_DIR /"c0005_alpha_q0_agg500_14000jets_5folds" / f"c0005_alpha_q0_agg500_14000jets_fold{f}_train.csv"
    test_csv  = AGG_METADATA_DIR /"c0005_alpha_q0_agg500_14000jets_5folds" / f"c0005_alpha_q0_agg500_14000jets_fold{f}_test.csv"

    fold_train.to_csv(train_csv, index=False)
    fold_test.to_csv(test_csv, index=False)

    print(
        f"Fold {f}: train={len(fold_train)}, test={len(fold_test)} "
        f"-> {train_csv.name}, {test_csv.name}"
    )


Fold 0: train=11200, test=2800 -> c0005_alpha_q0_agg500_14000jets_fold0_train.csv, c0005_alpha_q0_agg500_14000jets_fold0_test.csv
Fold 1: train=11200, test=2800 -> c0005_alpha_q0_agg500_14000jets_fold1_train.csv, c0005_alpha_q0_agg500_14000jets_fold1_test.csv
Fold 2: train=11200, test=2800 -> c0005_alpha_q0_agg500_14000jets_fold2_train.csv, c0005_alpha_q0_agg500_14000jets_fold2_test.csv
Fold 3: train=11200, test=2800 -> c0005_alpha_q0_agg500_14000jets_fold3_train.csv, c0005_alpha_q0_agg500_14000jets_fold3_test.csv
Fold 4: train=11200, test=2800 -> c0005_alpha_q0_agg500_14000jets_fold4_train.csv, c0005_alpha_q0_agg500_14000jets_fold4_test.csv
